In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("hwk3-task2-spills.ipynb")

# Task 2: Visualizing oil spills in NY State

## Instructions

- First, update the following cell to have a link to *your* Homework 3 GitHub repository:

**UPDATE THIS LINK**
https://github.com/kellyloveskelp/eds220-hwk3


- Review the [complete rubric for this task](https://docs.google.com/document/d/1Ce4EO3MEHW5uLks4x8BECWlkim2bcmxB4f8Qo4z26z8/edit?usp=sharing) before starting.

- **Meaningful commits should be made every time you finish a major step.** We'll check your repository and view the commit history.

- Comment mindfully in a way that enriches your code. Comments should follow best practices.

- **Do not update the top cell with the `otter` import**, this is used internally for grading.


## About the data
In this task you will use two datasets:

### First dataset: oil spills data

The first dataset contains information about [spilling incidents in the state of New York](https://data.ny.gov/Energy-Environment/Spill-Incidents/u44d-k5fk). Make sure you study the data portal and understand what each row represents before starting to code. 
You will find extra information and metadata for this dataset in the `NYSDEC_SpillIncidents_Overview.pdf` and `NYSDEC_SpillIncidents_DataDictionary.pdf` documents available at the portal.

### Second dataset: TIGER shapefile

The second dataset is the [2026 Counties (and equivalent) TIGER shapefile](https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.2026.html) (file `tl_2026_us_county.zip`) from the Census Bureau. You can check the [metadata for all the TIGER shapefiles here](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html). 

### COMPLETE WORKFLOW
You will use the next code cell to complete the last exercise in the task. Leave it blank for now.

In [ ]:
# FULL CODE

# Load libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

# Import data and filter
fp = os.path.join('data', 'tl_2026_us_county', 'tl_2026_us_county.shp')
counties = gpd.read_file(fp)


csv = os.path.join('data' , 'Spill_Incidents_20261107.csv')
spills = pd.read_csv(csv)
# Remove county labeled Canada to only query US counties
spills = spills[~spills['County'].str.contains('Canada', case=False)]

# Clean column names
spills.columns = spills.columns.str.lower().str.replace(' ', '_')

# Update spill_date to datetime
spills['spill_date'] = pd.to_datetime(spills['spill_date'])

# Select petroleum spills from Jan to Dec 2023
petr = spills[spills['material_family'] == ('Petroleum')]
petr_23 = petr[(petr['spill_date'] >= '2023-01-01') & (petr['spill_date'] <= '2023-12-31')]

# Group unique spill numbers by county
spills_per_county = pd.DataFrame(
    petr_23.groupby('county')
    ['spill_number'].nunique()
    .reset_index()
    .rename(columns={'spill_number':'n_spills'}))

# Clean column names in counties dataframe
counties.columns = counties.columns.str.lower()
counties = counties.rename(columns={
    'statefp': "state_fp",
    'countyfp':'county_fp',
    'countyns':'county_ns',
    'namelsad':'name_lsad'})

# Select counties in NY by state_fp ID
ny_counties = counties[counties['state_fp'] == '36']

# Rename "St. Lawrence" value
spills_per_county = spills_per_county.replace({'county': {'St Lawrence':'St. Lawrence'}})

# Rename columns to prepare for merge
spills_per_county = spills_per_county.rename(columns={'county':'county_name'})
ny_counties = ny_counties.rename(columns={'name':'county_name'})

# Inner Join dataframes
ny_spills = pd.merge(spills_per_county,
                     ny_counties, 
                     how = 'inner', 
                     on = 'county_name')

# Recasting `ny_spills` as a GeoDataFrame
ny_spills = gpd.GeoDataFrame(ny_spills, geometry="geometry")

# Add Indian Reservation spills to n_spills in nearby county
ny_spills.loc[31, 'n_spills'] = 97 + 2
ny_spills.loc[33, 'n_spills'] = 214 + 1

### Plot ny_spills ###
fig, ax = plt.subplots(figsize=(8,6))

ax.axis('off')

ny_spills.plot(ax=ax,
                column = 'n_spills',
                legend = True,
                edgecolor = "1.0",
                cmap = 'YlOrRd',
                legend_kwds ={
                    "label" : "Number of Spills",
                    "orientation" : "vertical"
                    }
                )

ax.set_title('Petroleum Spills by County in New York, 2023',
             fontsize=14, pad=15)

caption = 'New York State Department of Environmental Conservation, Date Accessed: Nov 7, 2026'
fig.text(0.5, 0.01, caption, wrap=True, horizontalalignment='center', fontsize=10, style='italic')


plt.show()

## 1. File management
a. Any data you download should go in the `data/` folder of your `eds220-hwk3` repository. For example, after downloading both datasets, your repository should look like this:

```
eds220-hwk3/
├── data/
│   ├── q5d_df.csv
│   ├── Spill_Incidents_20261027.csv
│   └── tl_2026_us_county/
│       ├── tl_2026_us_county.cpg
│       ├── tl_2026_us_county.dbf
│       ├── tl_2026_us_county.prj
│       ├── tl_2026_us_county.shp
│       └── ... (other files from the shapefile)
├── tests/
├── .gitignore
├── LICENSE
├── README.md
└── hwk3-task2-spills.ipynb
```

The date in the name of the spills file may be different, depending on when you download it. `data/q5d_df.csv` is already in the repository and is used by the autograder tests, so do not delete or move it.

**IMPORTANT: Do not commit your data!**

The datasets we'll be working with are large and will most likely jam your commit history if git does not ignore them.

b. Add the relative path of each dataset you downloaded to your `.gitignore` file. For example, to ignore the folder containing the shapefile, add `data/tl_2026_us_county/` to the `.gitignore`.

c. Commit and push your changes to the `.gitignore`.

d. Run `git status` and verify that none of your data is being tracked by git.

## 2. Datasets description
- Read the metadata for both datasets and write a brief description about each one. Indicate the data sources and date of access.

## 3. Import libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

## 4. Import data 
In the next cell:

- Import the NY spills data as a variable named `spills`. 
- Import the US counties data as a variable named `counties`.

If you are using file paths to access the data, these should be constructed using the `os` package.

In [ ]:
fp = os.path.join('data', 'tl_2026_us_county', 'tl_2026_us_county.shp')
counties = gpd.read_file(fp)


csv = os.path.join('data' , 'Spill_Incidents_20261107.csv')
spills = pd.read_csv(csv)

# Remove county labeled Canada to only query US counties
spills = spills[~spills['County'].str.contains('Canada', case=False)]

<!-- BEGIN QUESTION -->

## 5. Prepare spills data

This section refers to the `spills` dataframe. 
Check the outputs carefully, they will give you information you will need to use in the next exercises.

### 5.A. Exploration

In the next cells:

a. Check the dataframe's head

b. Check the data types of the columns

c. Check the unique values in the `material_family` column

d. Check the unique values in the `county` column

e. Insert a markdown cell to explain what information you obtained from the preliminary exploration. Use complete sentences.

You can add any other cells for preliminary data exploration.

_Type your answer here, replacing this text._

In [ ]:
#a. Check the dataframe's head
spills.head(3)

In [ ]:
#b. Check the data types of the columns
spills.dtypes

In [ ]:
#c. Check the unique values in the `material_family` column
spills['Material Family'].nunique

In [ ]:
#d. Check the unique values in the `county` column
spills['County'].nunique

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 5.B. Cleaning
a. Simplify column names as needed.

In [ ]:
spills.columns = spills.columns.str.lower().str.replace(' ', '_')
spills.columns

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Update column data types as needed.

In [ ]:
spills['spill_date'] = pd.to_datetime(spills['spill_date'])
print(spills.dtypes)

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 5.C. Data selection

a. Select data about petroleum spills that took place between January 1 and December 31, 2023 (including both dates). Store it in a variable named `petr_23`. The data selection should occur in a single line. You may add cells as needed.

In [ ]:
petr = spills[spills['material_family'] == ('Petroleum')]
petr_23 = petr[(petr['spill_date'] >= '2023-01-01') & (petr['spill_date'] <= '2023-12-31')]
petr_23['spill_date'].sort_values()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Use `print()` and f-strings to print a message verifying that `petr_23` only has data for 2023 and from January through December.
HINT: if you converted the `spill_date` column to datetime, you can use `petr_23['spill_date'].dt.year.unique()`.

In [ ]:
def date_check(df_col):
    """
        Check year and months contained in dataframe date column.
        
        Args:
            df_col: Column within dataframe to check.
            
        Returns:
            Boolean conditons and output messages.
        """
    year_range = 2023
    year = df_col.dt.year.unique()

    month_range = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] # Jan through Dec
    months = df_col.dt.month.sort_values().unique().tolist()

    if year == year_range:
        print(f"Dataframe contains data for the year(s) {year}.")
    if months == month_range:
        print("Dataframe contains data for all months from Jan to Dec.")
    else:
        print("Stop! Dataframe contains data outside desired range.")
    return

date_check(petr_23['spill_date'])

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

c. Write tests using `assert` to verify that `petr_23` only has data for petroleum spills.

In [ ]:
def date_check(df_col):
    """
        Check year and months contained in dataframe date column.
        
        Args:
            df_col: Column within dataframe to check.
            
        Returns:
            Warning messages.
        """
    year_range = 2023
    year = df_col.dt.year.unique()

    month_range = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12] # Jan through Dec
    months = df_col.dt.month.sort_values().unique().tolist()

    assert year == year_range, f"Dataframe must only contain data for 2023, got {year}"
    assert months == month_range, f"Dataframe must contain data for all months from Jan to Dec, got {months}"

    return

date_check(petr_23['spill_date'])

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

d. Use `print()` and f-strings to print the number of rows in `petr_23` and the number of unique spills (use the `spill_number` column). Are these numbers the same? In the next markdown cell, explain why this might be the case. Use complete sentences. You may use additional cells to explore the data if needed.

_The number of rows and unique spill numbers are not the same. This indicates there are rows with duplicate spill numbers, likely indicating that some spills spanned multiple counties or continuous dates and therefore were recorded multiple times. It may also be due to the same spill containing multiple unique values for attributes such as source, waterbody, or material names._

In [ ]:
print(f"There are {petr_23.shape[0]} rows in the `petr_23` dataset.")
print(f"And there are {petr_23['spill_number'].nunique()} unique spills in the `petr_23` dataset.")
print(f"Therefore, there are {(petr_23.shape[0]) - (petr_23['spill_number'].nunique())} extra rows with potential duplicate spill numbers.")

<!-- END QUESTION -->

### 5.D. Data aggregation

Create new dataframe named `spills_per_county` with the number of petroleum spills per county from January to December 2023.

As you found in exercise 5.C.d, a spill can appear in more than one row. Make sure you count each spill only once. HINT: `nunique()`.

The resulting `spills_per_county` dataframe must be as follows:

- Index: integer numbers starting from 0
- Two columns only.
- Column one: county names, column name = `county`
- Column two: number of petroleum spills per county in 2023, column name = `n_spills`

You may add cells if needed for your data wrangling.

In [ ]:
spills_per_county = pd.DataFrame(
    petr_23.groupby('county')
    ['spill_number'].nunique()
    .reset_index()
    .rename(columns={'spill_number':'n_spills'}))
spills_per_county


In [ ]:
grader.check("q5d")

<!-- BEGIN QUESTION -->

## 6. Prepare counties data

This section refers to the `counties` geo-dataframe. 
Check the outputs carefully, they will give you context about the next exercises.

### 6.A. Exploration

In the next cells:

a. Check the geo-dataframe's head

b. Check the data types of the columns

c. Check the geo-dataframe's CRS

d. Plot the geo-dataframe

e. Use the next markdown cell to explain what information you obtained from the preliminary exploration. Use complete sentences.

You can add any other cells of preliminary data exploration.

_Type your answer here, replacing this text._

In [ ]:
# a
counties.head()

In [ ]:
# b
counties.dtypes

In [ ]:
# c
counties.crs

In [ ]:
# d
counties.plot()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 6.B. Cleaning

Simplify column names as needed.

In [ ]:
counties.columns = counties.columns.str.lower()
counties = counties.rename(columns={
    'statefp': "state_fp",
    'countyfp':'county_fp',
    'countyns':'county_ns',
    'namelsad':'name_lsad'})
counties.columns

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### 6.C. Data selection

Select all the counties for the state of New York in a new variable `ny_counties` and plot them (you may need to look at the metadata).

In [ ]:
# Finding the `state_fp` ID for New York using New York County, New York
counties[counties['name'] == 'New York']['state_fp']

In [ ]:
ny_counties = counties[counties['state_fp'] == '36']
ny_counties['name'].nunique()

In [ ]:
# Did I capture NY ?
ny_counties.plot()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

## 7 Merging

The goal of this section is to create a single dataframe with the number of spills per county and the corresponding geometries for each county.

### 7.A. Explore data

In the next cells:

a. Print the names of counties in `ny_counties`, without repetition.


In [ ]:
ny_counties['name'].unique()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

b. Print the names of counties in `spills_per_county`, without repetition. Examine the differences in the outputs.

In [ ]:
spills_per_county['county'].nunique()

<!-- END QUESTION -->



c. Which are the county names that are in `spills_per_county`, but not in `ny_counties`? Your answer should be a list of strings
HINT: You can use the `numpy` function [`setdiff1d()`](https://numpy.org/doc/stable/reference/generated/numpy.setdiff1d.html), pay attention to the example in the documentation. Store your answer in the `diff_names` variable. 


In [ ]:
a = spills_per_county['county'].sort_values().unique()
b = ny_counties['name'].sort_values().unique()

diff_names_spills = np.setdiff1d(a, b)
diff_names_spills


In [ ]:
grader.check("7Ac")

d. Which are the county names that are in `ny_counties`, but not in `spills_per_county`? Your answer should be a list of a string/strings.


In [ ]:
diff_names_ny = np.setdiff1d(b, a)
diff_names_ny

In [ ]:
grader.check("7Ad")

e. What is the number of spills in 'Tuscarora Indian Reservation' and in 'Onondaga Indian Reservation'?

In [ ]:
# e. 
print(f"Tuscarora Indian Reservation: {spills_per_county[spills_per_county['county'] == 'Tuscarora Indian Reservation']['n_spills'].tolist()} spill(s)")
print(f"Onondaga Indian Reservation: {spills_per_county[spills_per_county['county'] == 'Onondaga Indian Reservation']['n_spills'].tolist()} spill(s)")

### 7.B. Data updates

In the next cell, update 'St Lawrence' to 'St. Lawrence' in the `spills_per_county` data frame. Verify the update was successful with an assert test. 


In [ ]:
# Pulling from original error message and instructor code
spills_per_county = spills_per_county.replace({'county': {'St Lawrence':'St. Lawrence'}})
spills_per_county[spills_per_county['county'] == 'St. Lawrence']

In [ ]:
assert spills_per_county.loc[52]['county'] == 'St. Lawrence', f"Rename 'county' to St. Lawrence, got St Lawrence."

### 7.C. Merge

In the following cells:

a. Use this cell to make any other updates needed for making an inner join of the `spills_per_county` and `ny_counties`. HINT: [`pd.merge`](https://pandas.pydata.org/pandas-docs/version/2.3/reference/api/pandas.merge.html)


In [ ]:
# a 
spills_per_county = spills_per_county.rename(columns={'county':'county_name'})
ny_counties = ny_counties.rename(columns={'name':'county_name'})

b. Store the inner join of the `spills_per_county` and `ny_counties` dataframes in a new variable named `ny_spills`.


In [ ]:
ny_spills = pd.merge(spills_per_county,
                     ny_counties, 
                     how = 'inner', 
                     on = 'county_name')

# Recasting `ny_spills` as a GeoDataFrame
ny_spills = gpd.GeoDataFrame(ny_spills, geometry="geometry")

<!-- BEGIN QUESTION -->

c. An inner join only keeps the rows whose `county` appears in both dataframes, and it does not warn you about the rows it drops. Which records from `spills_per_county` were dropped by the inner join, and how many spills did each one have? Add the cells you need to find the dropped records.

In [ ]:
c = spills_per_county['county_name'].sort_values().unique()
d = ny_spills['county_name'].sort_values().unique()

dropped_spills = np.setdiff1d(c, d)
dropped_spills

In [ ]:
dropped_cols = dropped_spills.tolist()
dropped_spills = spills_per_county[spills_per_county['county_name'].isin(dropped_cols)]
dropped_spills

In [ ]:
dropped_spills['n_spills'].sum()

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

d. Does dropping these records affect the map or the information we will report? Write your answer in the following markdown cell.

_Yes, dropping these records removes 13 spills from the visualization, including spills on two Indian Reservations. As a result, our output map will not display spills on Indigenous land._

<!-- END QUESTION -->


From our previous exploration we know that Tuscarora Indian Reservation and Onondaga Indian Reservation will not be in the join. Tuscarora Indian Reservation is located within Niagara county and Onondaga Indian Reservation is located within Onondaga county. 

e. Since the reservations do not appear in the `ny_counties` dataframe, add the Tuscarora Indian Reservation spills to Niagara county and the Onondaga Indian Reservation spills to Onondaga county in `ny_spills`. We will add a note about this in our final map, and we will reflect on this choice in exercise 8.b. Add the cells you need to make this update **and** verify that the dataframe was updated.

In [ ]:
print(ny_spills[ny_spills['county_name'] == "Niagara"]['n_spills'])
print(ny_spills[ny_spills['county_name'] == "Onondaga"]['n_spills'])

In [ ]:
# Add 2 to Niagara county with .loc()
ny_spills.loc[31, 'n_spills'] = 97 + 2

# Verify dataframe was updated
assert ny_spills.loc[31, 'n_spills'] == (97 + 2)

In [ ]:
# Add 1 to Onondaga county with .loc()
ny_spills.loc[33, 'n_spills'] = 214 + 1

# Verify dataframe was updated
assert ny_spills.loc[33, 'n_spills'] == (214 + 1)

## 8. Map
a. Create a choropleth map of the number of petroleum spills per county in the state of New York in 2023. 
Your map should be polished and have (at least) the following updates:
- a legend indicating the number of spills
- an updated color map
- a title 
- no axis
- an annotation with the data source (New York State Department of Environmental Conservation) and date of access



In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

ax.axis('off')

ny_spills.plot(ax=ax,
                column = 'n_spills',
                legend = True,
                edgecolor = "1.0",
                cmap = 'YlOrRd',
                legend_kwds ={
                    "label" : "Number of Spills",
                    "orientation" : "vertical"
                    }
                )

ax.set_title('Petroleum Spills by County in New York, 2023',
             fontsize=14, pad=15)

caption = 'New York State Department of Environmental Conservation, Date Accessed: Nov 7, 2026'
fig.text(0.5, 0.01, caption, wrap=True, horizontalalignment='center', fontsize=10, style='italic')


plt.show()

b. Write a description of how the map was created, including key takeaways. If you need to do additional data analysis to include relevant information from the data, you may include additional cells here.

In your description, also reflect on how we handled the spills reported in the Tuscarora and Onondaga Indian Reservations (exercise 7.C.e) in terms of data governance and the CARE Principles for Indigenous Data Governance from Task 1.

This map was created from two data sources: a record of petroleum and hazardous material spill incidents from [DATA.NY.GOV](https://data.ny.gov/Energy-Environment/Spill-Incidents/u44d-k5fk), and the U.S. Census Bureau [TIGER Data](https://www.census.gov/programs-surveys/geography/technical-documentation/complete-technical-documentation/tiger-geo-line.html). Spill incidents were filtered to petroleum spills from Janurary 1st to December 31st, 2023 and consolidated by county. It is important to note, there were nearly 200 duplicate spill incidences reported that were not included in final map. Precendence was given to counties higher in alphabetical order, therefore, spills per county in lower alphabetical order may be underestimated.

It is also important to note that spills on Indigenous land, two on the Tuscarora Indian Reservation and one on the Onondaga Indian Reservation, were lumped into the total local county spills in the final visualization. This decision was made because the TIGER shapefiles used to create `ny_spills` did not include polygon geometries for the reservations, therefore these spill counts would have been otherwise discluded. This is an attempt to minimize harm that would have been caused by discluding Indigenous data altogether, and maximize benefit to all who may use this data to advocate for themselves ([CARE Princples](https://static1.squarespace.com/static/5d3799de845604000199cd24/t/6397b1aff7a6fb54defdf687/1670885815820/dsj-1158_carroll.pdf)).

The map clearly indicates seven counties with distinctly high numbers of petroleum spills, as shown in the chart below. Westchester, Suffolk, and Nassau counties are the highest among them, diplayed in dark red.

In [ ]:
ny_spills.sort_values('n_spills', ascending = False).head(7)[['county_name', 'n_spills']]

Within Westchester, "Private Dwellings" are the most common source of spills, however, "Commercial/Industrial" sources are responsible for the highest quantity of petroleum spilled. One locale, Landmark at Eastview, headquarters for [Regeneron Pharmaceuticals](https://www.regeneron.com/about/our-company/locations), contributes 17% of the total petroleum spilled in the area by Commerical/Industrial sources. With concerted effort to prevent petroleum spills at this site, a major reduction in overall spills in Westchester could be made!

In [ ]:
spills[spills['county'] == "Westchester"]['source'].value_counts()

In [ ]:
spills = spills[(spills['material_family'] == "Petroleum") & (spills['units'] == "Gallons")]
spills['quantity'] = spills['quantity'].str.replace(",", "").astype('float')
spills[spills['county'] == "Westchester"].groupby('source')['quantity'].sum().sort_values(ascending = False)

In [ ]:
spills[(spills['county'] == "Westchester") & (spills['source'] == "Commercial/Industrial")
       ].groupby('program_facility_name')['quantity'].sum().sort_values(ascending = False).head()

## Complete workflow

Collect all the relevant code into the first blank cell of the notebook titled "COMPLETE WORKFLOW". This single cell will have the end-to-end workflow: from importing libraries and loading the data, to producing the choropleth map of the number of petroleum spills per county in the state of New York in 2023. The only output of this cell should be the choropleth map  you produced in the previous exercise. Further guidance on what to include in this final workflow is in the assignment rubric.